In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df = pd.read_csv("../gait-data/train_set.csv")
labels = pd.read_csv("../gait-data/labels_train_set.csv")
merged_df = pd.merge(train_df, labels, on="ID")
merged_df = merged_df.sort_values(by=["Subject", "Session", "Time:512Hz"])

In [3]:
from scripts.session_processing.sessions import iter_subject_sessions
from scripts.session_processing.preprocessing import (
    build_default_preprocessing_transforms,
    apply_transforms_over_subject_sessions
)

# feature_columns = ['FP1', 'FP2', 'F3', 'F4', 'FC1', 'FC2', 'C4', 'C3', 'Cz', 'CP2', 'CP1', 'P4', 'Pz', 'P3', 'O2', 'O1']
feature_columns = ['FP1', 'FP2', 'F3', 'F4', 'FC1', 'FC2', 'C4', 'C3', 'Cz', 'CP2', 'P4', 'Pz', 'P3', 'O2', 'O1'] # no CP1 cuz it looks corrupted
core_motor = ["C3", "Cz", "C4"]
motor_plus = ["FC1", "FC2", "C3", "Cz", "C4", "CP2"]


default_transforms = build_default_preprocessing_transforms(
    fs = 512.0,
    bandpass = (0.5, 45.0),
    notch_hz = None, 
    apply_car = False,
    apply_zscore = True,
    clip_uv = None
    )
preprocessed_sessions = apply_transforms_over_subject_sessions(
    merged_df,
    default_transforms,
    feature_cols=core_motor,
    target_col="angle",
)

In [5]:
preprocessed_sessions

,ID,Subject,Session,Time:512Hz,FP1,FP2,F3,F4,FC1,FC2,...,Cz,CP2,CP1,P4,Pz,P3,O2,O1,time,angle
0,0,SCI2,1,0.000000,-15006.495117,760.498108,-14070.362305,-1148.193481,-9447.022461,-16526.857422,...,0.113664,-10324.366211,-1.045147e+08,-23172.169922,-1313.183716,-14525.928711,-22304.298828,-20537.404297,0.000000,1.057401
1,1,SCI2,1,0.001953,-15006.446289,750.927795,-14067.481445,-1145.996216,-9448.047852,-16520.363281,...,0.061676,-10316.358398,-1.045147e+08,-23159.816406,-1312.304810,-14525.098633,-22321.095703,-20543.654297,0.001953,1.075232
2,2,SCI2,1,0.003906,-15025.928711,746.826233,-14080.225586,-1126.318481,-9455.274414,-16515.675781,...,0.027193,-10325.147461,-1.045147e+08,-23177.687500,-1317.431763,-14528.809570,-22313.722656,-20539.796875,0.003906,1.089365
3,3,SCI2,1,0.005859,-15005.372070,768.261780,-14069.483398,-1135.888794,-9457.618164,-16527.052734,...,0.024178,-10331.153320,-1.045147e+08,-23187.550781,-1326.904419,-14546.680664,-22341.408203,-20583.449219,0.005859,1.099761
4,4,SCI2,1,0.007812,-15004.834961,764.160217,-14071.338867,-1146.728638,-9451.661133,-16524.074219,...,0.059884,-10327.881836,-1.045146e+08,-23174.171875,-1325.390747,-14547.071289,-22339.552734,-20578.468750,0.007812,1.106410
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3048425,3048425,SCI3,14,127.990234,-7123.828613,-7253.809082,-6995.068848,1930.615356,2400.293213,26887.355469,...,-0.875151,-3174.267822,9.107276e+03,25294.630859,4371.191895,17035.304688,19428.273438,17763.625000,127.990234,124.382060
3048426,3048426,SCI3,14,127.992188,-7114.844238,-7255.762207,-6988.574707,1930.420044,2402.197510,26888.136719,...,-0.826153,-3173.291260,9.113087e+03,25292.970703,4370.850098,17040.138672,19427.150391,17771.486328,127.992188,124.408879
3048427,3048427,SCI3,14,127.994141,-7119.189941,-7254.736816,-6988.916504,1930.468872,2398.926025,26891.310547,...,-0.745904,-3176.220947,9.107228e+03,25294.630859,4367.676270,17036.134766,19420.119141,17766.066406,127.994141,124.410576
3048428,3048428,SCI3,14,127.996094,-7116.797363,-7256.152832,-6990.283691,1930.810669,2403.808838,26890.333984,...,-0.638037,-3177.148682,9.106300e+03,25287.648438,4365.088379,17033.009766,19420.851562,17756.496094,127.996094,124.386556


In [6]:
sci2_preprocessed = preprocessed_sessions[preprocessed_sessions['Subject'] == 'SCI2']
sci3_preprocessed = preprocessed_sessions[preprocessed_sessions['Subject'] == 'SCI3']

In [10]:
# create 1-second windows with a 0.1-second stride and a 0.1-second target lag
from scripts.session_processing.windowing import build_windows_over_subject_session


X_all, y_all, meta = build_windows_over_subject_session(
    sci2_preprocessed,
    feature_cols=core_motor,
    target_col="angle", 
    fs=512,
    window_s=1.0,
    stride_s=0.1,
    lag_s=0.0,
    return_meta=True
)

In [12]:
import torch
import torch.nn as nn

class CNNLSTM(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        conv_channels=(32, 64, 128),
        kernel_size=3,
        dropout=0.2,
        output_size=1,
    ):
        super().__init__()

        c1, c2, c3 = conv_channels
        pad = kernel_size // 2

        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, c1, kernel_size=kernel_size, padding=pad),
            nn.BatchNorm1d(c1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # seq_len -> seq_len/2

            nn.Conv1d(c1, c2, kernel_size=kernel_size, padding=pad),
            nn.BatchNorm1d(c2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # seq_len -> seq_len/4

            nn.Conv1d(c2, c3, kernel_size=kernel_size, padding=pad),
            nn.BatchNorm1d(c3),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(
            input_size=c3,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, input_size, seq_len)
        x = self.cnn(x)           # (batch, c3, reduced_seq_len)
        x = x.transpose(1, 2)     # (batch, reduced_seq_len, c3)
        x, _ = self.lstm(x)       # (batch, reduced_seq_len, hidden_size)
        x = x[:, -1, :]           # last timestep
        x = self.dropout(x)
        x = self.fc(x)            # (batch, output_size)
        return x

In [13]:
# set up the data
from torch.utils.data import DataLoader, TensorDataset


X = torch.tensor(X_all, dtype=torch.float32)
y = torch.tensor(y_all, dtype=torch.float32).unsqueeze(1)  # (num_samples, 1)

X = X.transpose(1, 2)  # (num_samples, num_features, seq_len)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [14]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [15]:
model = CNNLSTM(input_size=X.shape[1], hidden_size=64, num_layers=2).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

import time

# track losses for plotting
train_losses = []

# early stopping
patience = 5
epochs_no_improve = 0
best_loss = float("inf")

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    start = time.time()
    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)

    epoch_loss /= len(dataset)
    train_losses.append(epoch_loss)
    elapsed = time.time() - start

    print(f"Epoch {epoch+1}/{num_epochs} — Loss: {epoch_loss:.4f} ({elapsed:.0f}s)")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  → saved best model")
    else:
        epochs_no_improve += 1
        print(f"  → no improvement ({epochs_no_improve}/{patience})")
        if epochs_no_improve >= patience:
            print("Early stopping!")
            break

Epoch 1/50 — Loss: 2140.3955 (15s)
  → saved best model
Epoch 2/50 — Loss: 2139.3332 (13s)
  → saved best model
Epoch 3/50 — Loss: 2139.2143 (13s)
  → saved best model
Epoch 4/50 — Loss: 2137.1289 (13s)
  → saved best model
Epoch 5/50 — Loss: 2139.0764 (13s)
  → no improvement (1/5)
Epoch 6/50 — Loss: 2138.1540 (14s)
  → no improvement (2/5)
Epoch 7/50 — Loss: 2138.2416 (13s)
  → no improvement (3/5)
Epoch 8/50 — Loss: 2137.6990 (13s)
  → no improvement (4/5)
Epoch 9/50 — Loss: 2136.2771 (13s)
  → saved best model
Epoch 10/50 — Loss: 2133.4807 (13s)
  → saved best model
Epoch 11/50 — Loss: 2134.0004 (13s)
  → no improvement (1/5)
Epoch 12/50 — Loss: 2132.7769 (13s)
  → saved best model
Epoch 13/50 — Loss: 2132.0727 (14s)
  → saved best model
Epoch 14/50 — Loss: 2129.5130 (15s)
  → saved best model
Epoch 15/50 — Loss: 2127.2646 (14s)
  → saved best model
Epoch 16/50 — Loss: 2127.0646 (14s)
  → saved best model
Epoch 17/50 — Loss: 2123.3740 (14s)
  → saved best model
Epoch 18/50 — Loss: 

## The following cells are debugging the model on a small dataset to see if it can learn the data at all (will overfit aggresivly)

In [15]:
# overfit sanity check: one session through the current preprocessing + windowing pipeline
debug_subject = "SCI2"
debug_session = 1
debug_window_s = 1.0
debug_stride_s = 0.1
debug_lag_s = 0.1
debug_subset_size = 512

debug_session_df = (
    merged_df[(merged_df["Subject"] == debug_subject) & (merged_df["Session"] == debug_session)]
    .sort_values("Time:512Hz")
    .reset_index(drop=True)
)

debug_X, debug_y, debug_meta = build_windows_over_subject_session(
    debug_session_df,
    feature_cols=feature_columns,
    target_col="angle",
    fs=512,
    window_s=debug_window_s,
    stride_s=debug_stride_s,
    lag_s=debug_lag_s,
    return_meta=True,
)

if len(debug_X) == 0:
    raise ValueError("No debug windows were created. Check the selected session and window settings.")

debug_subset_size = min(debug_subset_size, len(debug_X))
debug_X = debug_X[:debug_subset_size]
debug_y = debug_y[:debug_subset_size].astype(np.float32)
debug_meta = debug_meta.iloc[:debug_subset_size].reset_index(drop=True)

debug_X = debug_X.transpose(0, 2, 1).astype(np.float32)

debug_baseline_mse = float(np.mean((debug_y - debug_y.mean()) ** 2))

print(f"Debug session: {debug_subject} / {debug_session}")
print(f"Debug windows: {debug_X.shape}")
print(f"Debug targets: {debug_y.shape}")
print(f"Debug lag (s): {debug_lag_s}")
print(f"Mean-baseline MSE on debug subset: {debug_baseline_mse:.4f}")

Debug session: SCI2 / 1
Debug windows: (512, 16, 512)
Debug targets: (512,)
Debug lag (s): 0.1
Mean-baseline MSE on debug subset: 82.3003


In [17]:
from torch.utils.data import DataLoader, TensorDataset

debug_device = torch.device("mps" if torch.mps.is_available() else "cpu")

debug_dataset = TensorDataset(
    torch.tensor(debug_X, dtype=torch.float32),
    torch.tensor(debug_y, dtype=torch.float32).unsqueeze(1),
)
debug_loader = DataLoader(
    debug_dataset,
    batch_size=min(64, len(debug_dataset)),
    shuffle=True,
)

debug_model = CNNLSTM(input_size=debug_X.shape[1], hidden_size=64, num_layers=2).to(debug_device)
debug_criterion = nn.MSELoss()
debug_optimizer = torch.optim.Adam(debug_model.parameters(), lr=1e-3)

print(debug_device)
print(f"Debug model parameters: {sum(p.numel() for p in debug_model.parameters()):,}")

mps
Debug model parameters: 115,937


In [18]:
debug_epochs = 100
debug_train_losses = []

for epoch in range(1, debug_epochs + 1):
    debug_model.train()
    debug_total_loss = 0.0

    for xb, yb in debug_loader:
        xb, yb = xb.to(debug_device), yb.to(debug_device)

        debug_optimizer.zero_grad()
        debug_pred = debug_model(xb)
        debug_loss = debug_criterion(debug_pred, yb)
        debug_loss.backward()
        debug_optimizer.step()

        debug_total_loss += debug_loss.item() * xb.size(0)

    debug_epoch_loss = debug_total_loss / len(debug_dataset)
    debug_train_losses.append(debug_epoch_loss)

    if epoch == 1 or epoch % 10 == 0 or epoch == debug_epochs:
        print(
            f"Epoch {epoch:03d} | train MSE: {debug_epoch_loss:.4f} | "
            f"baseline MSE: {debug_baseline_mse:.4f}"
        )

Epoch 001 | train MSE: 82.4848 | baseline MSE: 82.3003
Epoch 010 | train MSE: 48.9328 | baseline MSE: 82.3003
Epoch 020 | train MSE: 19.3854 | baseline MSE: 82.3003
Epoch 030 | train MSE: 10.5575 | baseline MSE: 82.3003
Epoch 040 | train MSE: 5.6359 | baseline MSE: 82.3003
Epoch 050 | train MSE: 3.8837 | baseline MSE: 82.3003
Epoch 060 | train MSE: 3.3969 | baseline MSE: 82.3003
Epoch 070 | train MSE: 3.1112 | baseline MSE: 82.3003
Epoch 080 | train MSE: 2.7201 | baseline MSE: 82.3003
Epoch 090 | train MSE: 2.3870 | baseline MSE: 82.3003
Epoch 100 | train MSE: 1.7893 | baseline MSE: 82.3003


In [ ]:
import matplotlib.pyplot as plt

debug_model.eval()
with torch.no_grad():
    debug_pred_all = debug_model(
        torch.tensor(debug_X, dtype=torch.float32, device=debug_device)
    ).cpu().squeeze(1).numpy()

debug_fit_mse = float(np.mean((debug_pred_all - debug_y) ** 2))
print(f"Final debug train MSE: {debug_fit_mse:.4f}")
print(f"Improvement over mean baseline: {debug_baseline_mse - debug_fit_mse:.4f}")

plot_n = min(200, len(debug_y))
plt.figure(figsize=(10, 4))
plt.plot(debug_train_losses)
plt.axhline(debug_baseline_mse, color="red", linestyle="--", label="mean baseline")
plt.title("Current pipeline overfit check: train MSE")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(debug_y[:plot_n], label="target", alpha=0.8)
plt.plot(debug_pred_all[:plot_n], label="prediction", alpha=0.8)
plt.legend()
plt.title(f"Current pipeline overfit check: {debug_subject} / {debug_session}")
plt.xlabel("window index")
plt.ylabel("angle")
plt.show()